# Rule of Thumb — tabular quickstart

This notebook demonstrates the tabular RoT explainer on a small synthetic
problem: we define a toy "black box" classifier, fit a Rule-of-Thumb
surrogate to its outputs, and inspect the learned feature importances.

Everything runs on CPU with synthetic data in a few seconds.

In [ ]:
import numpy as np

from ruleofthumb import RuleOfThumb

In [ ]:
# Synthetic data and a trivially simple "black box"
rng = np.random.RandomState(0)
n, d = 2000, 5
X = rng.randn(n, d).astype(np.float32)

def black_box(X):
    # logistic rule driven by features 0 and 2 only
    z = 2.0 * X[:, 0] - 1.5 * X[:, 2]
    return (1 / (1 + np.exp(-z)) > 0.5).astype(np.int64)  # int labels

y = black_box(X)

In [ ]:
rot = RuleOfThumb(y_outputs=y, x_inputs=X, epochs=50, batch_size=500, learning_rate=0.05)
importances = rot.get_explanation(X)
importances[:3]

In [ ]:
# Mean importance per feature — features 0 and 2 should dominate
mean_imp = np.abs(importances).mean(axis=0)
for i, v in enumerate(mean_imp):
    print(f"feature {i}: {v:.4f}")

## Fidelity vs. number of revealed features

`score_ordering` reveals features most-important-first and measures how well
the partially-revealed RoT reproduces the black box's labels.

In [ ]:
import torch

x_t = torch.from_numpy(X)
y_t = torch.from_numpy(y.astype(np.int64))
order = rot._explainer_model.get_order(x_t)
acc = rot._explainer_model.score_ordering(x_t, y_t, order)
print("accuracy after revealing k=1..d features:")
print(acc.numpy())